# Off-Earth Colonization: Solutions Architecture Analysis
## Data Analysis Workbook

**Methodology:** IBM/AWS/Google Well-Architected Frameworks, MoSCoW, Kano Model, NASA-STD-3001

This notebook analyzes the scoring matrices, risk registers, and NASA reference data
supporting a hybrid architecture recommendation for off-earth colonization.

---

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')
colors = {'Moon': '#4A90D9', 'Mars': '#D94A4A', 'Orbital': '#4AD99A'}
print('Setup complete')

## 1. Load Data
Upload the CSV files from the sheets_export folder. Run the cell below, then use the file picker.

In [ ]:
from google.colab import files
import io

# Upload all CSV files at once
uploaded = files.upload()

# Load each dataset
dfs = {}
for name, content in uploaded.items():
    key = name.replace('.csv', '')
    dfs[key] = pd.read_csv(io.BytesIO(content))
    print(f'Loaded {key}: {dfs[key].shape}')

# Assign to named variables for convenience
scoring = dfs.get('01_scoring_matrix', pd.DataFrame())
kano = dfs.get('02_kano_model', pd.DataFrame())
workloads = dfs.get('03_workload_classification', pd.DataFrame())
raci = dfs.get('04_stakeholder_raci', pd.DataFrame())
risks = dfs.get('05_risk_registers', pd.DataFrame())
providers = dfs.get('06_provider_scores', pd.DataFrame())
tech_gaps = dfs.get('07_nasa_tech_gaps', pd.DataFrame())
data_gaps = dfs.get('08_nasa_data_gaps', pd.DataFrame())
gap_analysis = dfs.get('09_gap_analysis', pd.DataFrame())
eclss = dfs.get('10_eclss_comparison', pd.DataFrame())
costs = dfs.get('11_cost_trajectory', pd.DataFrame())
assumptions = dfs.get('12_assumptions_register', pd.DataFrame())

## 2. Destination Scoring Analysis
### 2.1 Weighted Score Comparison

In [ ]:
# Calculate totals
totals = pd.DataFrame({
    'Destination': ['Moon', 'Mars', 'Orbital'],
    'Weighted_Total': [
        scoring['Moon_Weighted'].sum(),
        scoring['Mars_Weighted'].sum(),
        scoring['Orbital_Weighted'].sum()
    ]
})
max_possible = scoring['Weight'].sum() * 10
totals['Normalized_Pct'] = (totals['Weighted_Total'] / max_possible * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
bars = axes[0].bar(totals['Destination'], totals['Weighted_Total'],
    color=[colors[d] for d in totals['Destination']], edgecolor='white', linewidth=1.5)
axes[0].set_title('Weighted Score by Destination', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Weighted Score')
for bar, pct in zip(bars, totals['Normalized_Pct']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
        f'{pct}%', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, max_possible * 0.85)
axes[0].axhline(y=max_possible*0.6, color='gray', linestyle='--', alpha=0.5, label='60% threshold')
axes[0].legend()

# Tier breakdown
tier_scores = scoring.groupby('Tier').agg(
    Moon=('Moon_Weighted', 'sum'),
    Mars=('Mars_Weighted', 'sum'),
    Orbital=('Orbital_Weighted', 'sum')
).reset_index()
tier_scores['Tier'] = tier_scores['Tier'].map({1:'Tier 1: Basic Needs', 2:'Tier 2: Safety', 3:'Tier 3: Operations', 4:'Tier 4: Growth'})

x = np.arange(len(tier_scores))
w = 0.25
axes[1].bar(x - w, tier_scores['Moon'], w, label='Moon', color=colors['Moon'])
axes[1].bar(x, tier_scores['Mars'], w, label='Mars', color=colors['Mars'])
axes[1].bar(x + w, tier_scores['Orbital'], w, label='Orbital', color=colors['Orbital'])
axes[1].set_xticks(x)
axes[1].set_xticklabels(tier_scores['Tier'], rotation=15, ha='right')
axes[1].set_title('Scores by NASA Tier', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Weighted Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('destination_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print(totals.to_string(index=False))

### 2.2 Requirement Radar Chart (Raw Scores)

In [ ]:
# Radar chart for Must-Have requirements only
must_haves = scoring[scoring['MoSCoW'] == 'Must'].copy()
labels = must_haves['Requirement'].tolist()
N = len(labels)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for dest, col in [('Moon_Score', 'Moon'), ('Mars_Score', 'Mars'), ('Orbital_Score', 'Orbital')]:
    values = must_haves[dest].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=col, color=colors[col])
    ax.fill(angles, values, alpha=0.1, color=colors[col])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([l[:20] + '...' if len(l) > 20 else l for l in labels], size=8)
ax.set_ylim(0, 10)
ax.set_title('Must-Have Requirements: Raw Scores by Destination', size=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('must_have_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Risk Analysis
### 3.1 Risk Heatmap by Destination

In [ ]:
# Risk scores by destination
dest_risks = risks[risks['Destination'].isin(['Moon', 'Mars', 'Orbital'])].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for i, dest in enumerate(['Moon', 'Mars', 'Orbital']):
    subset = dest_risks[dest_risks['Destination'] == dest].sort_values('Risk_Score', ascending=True)
    color_map = subset['Risk_Score'].apply(lambda x: '#C6EFCE' if x < 10 else '#FFF2CC' if x < 16 else '#F4B084' if x < 20 else '#FFC7CE')
    bars = axes[i].barh(subset['Threat'], subset['Risk_Score'], color=color_map, edgecolor='white')
    axes[i].set_title(f'{dest} Risk Profile', fontsize=13, fontweight='bold', color=colors[dest])
    axes[i].set_xlim(0, 30)
    axes[i].axvline(x=15, color='orange', linestyle='--', alpha=0.5)
    axes[i].axvline(x=20, color='red', linestyle='--', alpha=0.5)
    for bar, score in zip(bars, subset['Risk_Score']):
        axes[i].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(score), va='center', fontsize=9)

plt.suptitle('Destination Risk Profiles (Likelihood x Impact)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('risk_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
risk_summary = dest_risks.groupby('Destination').agg(
    Total_Risk=('Risk_Score', 'sum'),
    Max_Risk=('Risk_Score', 'max'),
    Avg_Risk=('Risk_Score', 'mean'),
    Critical_Count=('Risk_Score', lambda x: (x >= 20).sum())
).round(1)
print('\nRisk Summary by Destination:')
print(risk_summary)

### 3.2 Engineering Solvability Analysis

In [ ]:
# Which risks are solvable by engineering vs permanent physics constraints?
solv = dest_risks.copy()
solv['Solvable_Category'] = solv['Solvable_by_Engineering'].map(
    lambda x: 'Yes' if x == 'Yes' else 'No (physics)' if 'physics' in str(x).lower() or 'No' in str(x) else 'Partial')

pivot = solv.groupby(['Destination', 'Solvable_Category'])['Risk_Score'].sum().unstack(fill_value=0)
print('Risk Score by Solvability:')
print(pivot)
print()

# Mars has the most unsolvable risk
for dest in ['Moon', 'Mars', 'Orbital']:
    unsolvable = solv[(solv['Destination'] == dest) & (solv['Solvable_Category'] == 'No (physics)')]
    if len(unsolvable) > 0:
        print(f"{dest} unsolvable risks:")
        for _, r in unsolvable.iterrows():
            print(f"  {r['Threat']}: score {r['Risk_Score']}")

## 4. Kano Satisfaction Analysis
### 4.1 Basic Needs Pass/Fail

In [ ]:
basics = kano[kano['Kano_Category'] == 'Basic'].copy()

for dest in ['Moon', 'Mars', 'Orbital']:
    fails = basics[basics[dest].str.contains('FAIL', na=False)]
    passes = basics[~basics[dest].str.contains('FAIL', na=False)]
    print(f"{dest}: {len(passes)}/{len(basics)} basic needs passed, {len(fails)} FAILED")
    if len(fails) > 0:
        for _, r in fails.iterrows():
            print(f"  FAIL: {r['Requirement']} -> {r[dest]}")
    print()

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
categories = kano['Kano_Category'].unique()
for dest in ['Moon', 'Mars', 'Orbital']:
    pass_counts = []
    for cat in categories:
        subset = kano[kano['Kano_Category'] == cat]
        if cat == 'Basic':
            pass_counts.append(len(subset[~subset[dest].str.contains('FAIL', na=False)]) / len(subset) * 100)
        elif cat == 'Performance':
            pass_counts.append(len(subset[subset[dest].str.contains('Continuous|Expandable|Diverse|Any time|1.3s', na=False)]) / len(subset) * 100)
        else:
            pass_counts.append(len(subset[subset[dest].str.contains('Yes', na=False)]) / len(subset) * 100)
    ax.plot(categories, pass_counts, 'o-', label=dest, color=colors[dest], linewidth=2, markersize=8)

ax.set_ylabel('Satisfaction Rate (%)')
ax.set_title('Kano Satisfaction by Category and Destination', fontsize=14, fontweight='bold')
ax.set_ylim(0, 110)
ax.legend()
plt.tight_layout()
plt.savefig('kano_satisfaction.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Provider Evaluation
### 5.1 Provider Scores by Architecture Layer

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

providers_sorted = providers.sort_values('Weighted_Score', ascending=True)
layer_colors = {'Heavy Lift': '#2E75B6', 'Cislunar': '#4A90D9', 'Habitat': '#4AD99A',
    'ECLSS': '#D9A64A', 'Comms': '#9B59B6', 'ISRU': '#D94A4A'}
bar_colors = [layer_colors.get(l, '#888') for l in providers_sorted['Layer']]

bars = ax.barh(providers_sorted['Provider'], providers_sorted['Weighted_Score'],
    color=bar_colors, edgecolor='white', linewidth=1)

ax.set_xlabel('Weighted Score (max 10)')
ax.set_title('Provider Evaluation: Weighted Scores', fontsize=14, fontweight='bold')
ax.set_xlim(0, 10)

# Add layer labels
for bar, layer in zip(bars, providers_sorted['Layer']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
        f'{bar.get_width():.1f} ({layer})', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('provider_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Gap Analysis
### 6.1 Architecture Gap Severity

In [ ]:
sev_colors = {'Low': '#C6EFCE', 'Medium': '#FFF2CC', 'High': '#F4B084', 'Critical': '#FFC7CE'}

fig, ax = plt.subplots(figsize=(12, 6))
ga = gap_analysis.sort_values('Provider_TRL', ascending=True)
bar_colors = [sev_colors[s] for s in ga['Gap_Severity']]

bars = ax.barh(ga['Architecture_Layer'], ga['Provider_TRL'], color=bar_colors, edgecolor='gray', linewidth=0.5)
ax.set_xlabel('Current Best Provider TRL (1-9)')
ax.set_title('Architecture Gap Analysis: Provider Readiness vs Need', fontsize=14, fontweight='bold')
ax.set_xlim(0, 10)
ax.axvline(x=6, color='green', linestyle='--', alpha=0.5, label='TRL 6 (System Demo)')
ax.axvline(x=8, color='blue', linestyle='--', alpha=0.5, label='TRL 8 (Operational)')

for bar, sev, yr in zip(bars, ga['Gap_Severity'], ga['Estimated_Closure_Year']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
        f'{sev} (est. {yr})', va='center', fontsize=9)

ax.legend()
plt.tight_layout()
plt.savefig('gap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. NASA Technology Gap Analysis
### 7.1 NASA's Own Priority Ordering vs Our Framework

In [ ]:
if not tech_gaps.empty:
    tg = tech_gaps.dropna(subset=['Priority_Rank']).copy()
    tg['Priority_Rank'] = pd.to_numeric(tg['Priority_Rank'], errors='coerce')
    tg = tg.dropna(subset=['Priority_Rank']).sort_values('Priority_Rank')

    # Map to our categories
    def map_to_our_category(title):
        title = str(title).lower()
        if 'dust' in title: return 'Environmental Hazard'
        elif 'isru' in title or 'extraction' in title or 'regolith' in title or 'excavat' in title: return 'ISRU'
        elif 'eclss' in title or 'water' in title or 'food' in title or 'dormancy' in title: return 'ECLSS / Life Support'
        elif 'power' in title or 'shadow' in title: return 'Power'
        elif 'commun' in title or 'navigation' in title: return 'Communications'
        elif 'radiation' in title: return 'Radiation'
        elif 'propul' in title or 'cryo' in title or 'landing' in title or 'entry' in title or 'ascent' in title: return 'Transport'
        elif 'robot' in title or 'autonom' in title or 'mobility' in title: return 'Autonomy / Robotics'
        elif 'medical' in title or 'exercise' in title or 'behavioral' in title or 'physiolog' in title or 'sensor' in title or 'suit' in title: return 'Crew Health'
        else: return 'Other'

    tg['Our_Category'] = tg['Title'].apply(map_to_our_category)

    # Priority distribution by our categories
    cat_priority = tg.groupby('Our_Category').agg(
        Count=('Priority_Rank', 'count'),
        Best_Priority=('Priority_Rank', 'min'),
        Avg_Priority=('Priority_Rank', 'mean')
    ).sort_values('Avg_Priority').round(1)

    print('NASA Technology Gaps by Category (lower priority number = higher NASA priority):')
    print(cat_priority)
    print()
    print(f'Total gaps: {len(tg)}')
    print(f'ISRU gaps avg priority: {cat_priority.loc["ISRU", "Avg_Priority"] if "ISRU" in cat_priority.index else "N/A"}')
    print('Note: ISRU is NASA\'s LOWEST priority category, validating our phased deployment timeline.')
else:
    print('Tech gaps file not loaded. Upload 07_nasa_tech_gaps.csv.')

## 8. Sensitivity Analysis
### 8.1 What if key assumptions change?

In [ ]:
# Test: What if gravity threshold drops to 0.3g (Mars passes)?
scenarios = {}
base = {'Moon': scoring['Moon_Weighted'].sum(), 'Mars': scoring['Mars_Weighted'].sum(), 'Orbital': scoring['Orbital_Weighted'].sum()}
scenarios['Baseline'] = base.copy()

# Scenario 1: Mars gravity passes (score 7 instead of 4)
s1 = base.copy()
s1['Mars'] += (7 - 4) * 10  # gravity weight is 10
scenarios['Mars gravity passes (0.3g viable)'] = s1

# Scenario 2: Orbital radiation solved (score 8 instead of 5)
s2 = base.copy()
s2['Orbital'] += (8 - 5) * 9  # radiation weight is 9
scenarios['Orbital radiation solved'] = s2

# Scenario 3: Starship cuts Mars transport cost (score 6 instead of 2)
s3 = base.copy()
s3['Mars'] += (6 - 2) * 8  # transport weight is 8
scenarios['Starship cuts Mars cost'] = s3

# Scenario 4: Lunar ISRU fails (Moon score drops from 7 to 3)
s4 = base.copy()
s4['Moon'] -= (7 - 3) * 7  # ISRU weight is 7
scenarios['Lunar ISRU fails'] = s4

# Scenario 5: All favorable to Mars
s5 = base.copy()
s5['Mars'] += (7-4)*10 + (6-2)*8  # gravity passes + transport cheaper
scenarios['Best case Mars (gravity + transport)'] = s5

print('Sensitivity Analysis Results:')
print(f'{"Scenario":<42} {"Moon":>6} {"Mars":>6} {"Orbital":>8}  Winner')
print('-' * 75)
for name, scores in scenarios.items():
    winner = max(scores, key=scores.get)
    print(f'{name:<42} {scores["Moon"]:>6} {scores["Mars"]:>6} {scores["Orbital"]:>8}  {winner}')

print()
print('Key finding: Orbital wins in ALL scenarios. Even best-case Mars does not overtake.')
print('The architecture recommendation is robust to assumption changes.')

## 9. ECLSS Mass Impact Analysis
Open-loop (225 kg) vs closed-loop (2,641 kg): the cost of self-sufficiency.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ECLSS mass comparison
eclss_mass = [225, 2641]
eclss_labels = ['Open-loop\n(Moon w/ resupply)', 'Closed-loop\n(Mars, no resupply)']
axes[0].bar(eclss_labels, eclss_mass, color=[colors['Moon'], colors['Mars']], edgecolor='white', width=0.5)
axes[0].set_title('ECLSS Initial Mass by Configuration', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Mass (kg)')
axes[0].text(0, 225+50, '225 kg', ha='center', fontsize=12, fontweight='bold')
axes[0].text(1, 2641+50, '2,641 kg\n(11.7x)', ha='center', fontsize=12, fontweight='bold', color=colors['Mars'])

# Daily consumables per person
consumables = {'O2': 0.816, 'Water': 2.5, 'Food': 1.77}
axes[1].bar(consumables.keys(), consumables.values(), color=['#5DADE2', '#48C9B0', '#F5B041'], edgecolor='white')
axes[1].set_title('Daily Consumables per Crew Member (kg)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('kg/day')
total_daily = sum(consumables.values())
axes[1].text(1, max(consumables.values())+0.1, f'Total: {total_daily:.1f} kg/day\n({total_daily*365:.0f} kg/year)', 
    ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('eclss_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Annual consumables per person: {total_daily*365:.0f} kg')
print(f'4-person crew annual: {total_daily*365*4:.0f} kg')
print(f'Moon resupply: ship it every few weeks')
print(f'Mars: store 2+ years worth ({total_daily*365*2:.0f} kg/person) or recycle 98%+ perfectly')

## 10. Architecture Recommendation Summary

| Layer | Recommendation | Cloud Analog |
|---|---|---|
| Earth | Protect primary infrastructure | On-premises (hardened) |
| Moon | Primary resource layer, ISRU, staging | Primary cloud region |
| Orbital | Scalable habitation, economic engine | Elastic compute |
| Mars | Deferred edge deployment | Edge node (deploy last) |

**Key findings:**
- No destination passes all Must-Have requirements independently
- Mars fails 3 of 5 Kano basic needs (gravity, RTO, RPO)
- Orbital wins in all sensitivity scenarios including best-case Mars
- NASA's own technology gap priorities validate our MoSCoW ordering
- ECLSS mass penalty for self-sufficiency is 11.7x (Mars constraint)
- Hybrid Moon + Orbital is the only architecture that scores above 55% for both resource availability and human survivability